# RSNA Knee MRI — ResNet34 CNN training

A separate end-to-end CNN experiment. It borrows learning-rate groups, best-checkpoint saving and scheduling from the BirdCLEF notebooks; MRI preparation and label handling are specific to this competition.

Attach **gany24558/rsna-knee-normalized-training-data** (all completed shards) and the original competition data. Generated labels are already bundled. Enable a **GPU** and **Internet** for ImageNet weights and final Kaggle Model upload. If using attached ResNet34 ImageNet weights, set PRETRAINED_PATH. Do not attach DINO feature caches: CNN weights change during training.

The split holds out 20% of patient groups (or study groups if patient identities are unavailable). Only verified labels in held-out groups determine validation loss and AUROC. All labels in those groups are excluded from training. Early stopping: **three consecutive epochs without strictly lower validation loss**. The best epoch is exported, not the final epoch.

Upload destination: **gany24558/gc-rsna-knee-resnet34 / PyTorch / study-mil**. The final cell automatically uploads the real trained package when run. This notebook does not launch training until you run it on Kaggle. Keep the notebook/model private. The existing DINO submission notebook cannot load CNN weights; a CNN-specific inference adapter is needed later.


In [ ]:
from pathlib import Path
DATASET_ROOT = None
COMPETITION_ROOT = None
PRETRAINED_PATH = None # Optional attached official torchvision resnet34 state dictionary
RESUME_CHECKPOINT = None # Your own trusted last.pt from an earlier identical run
OUTPUT_ROOT = Path('/kaggle/working/rsna-knee-resnet34-training')
REQUIRE_GPU = True
UPLOAD_MODEL = True
MODEL_HANDLE = 'gany24558/gc-rsna-knee-resnet34/pyTorch/study-mil'
CFG = dict(seed=42,val_fraction=.2,image_size=224,slices_per_series=4,max_train_series=4,
           microbatch=8,accumulate=4,dropout=.2,backbone_lr=1e-5,head_lr=1e-4,
           epochs=35,patience=3,max_hours=7.5)


## Runtime modules
These are embedded; no repository checkout or separate Python files are required on Kaggle.

In [ ]:
import sys, json, hashlib, random
SOURCE_FILES = {'knee_data.py': '"""Offline RSNA frozen-DINO training runtime. Embedded verbatim in the notebook."""\nimport os\nos.environ[\'HF_HUB_OFFLINE\'] = \'1\'\nos.environ[\'TRANSFORMERS_OFFLINE\'] = \'1\'\nimport copy, gc, hashlib, io, json, math, random, shutil, tarfile, time\nfrom contextlib import nullcontext\nfrom pathlib import Path, PurePosixPath\nimport numpy as np\nimport pandas as pd\nimport torch\nfrom torch import nn\nfrom torch.nn import functional as F\nfrom sklearn.metrics import roc_auc_score, average_precision_score\n\nID, SID = \'StudyInstanceUID\', \'SeriesInstanceUID\'\nTARGETS = [\'ACL\', \'MCL\', \'Medial Meniscus\', \'Lateral Meniscus\', \'Medial OA\',\n           \'Lateral OA\', \'PF OA\', \'Effusion\', \'Synovitis\', "Baker\'s", \'Contusion\', \'Fracture\']\nRUNTIME_VERSION = \'rsna-frozen-dino-v1\'\n\n\ndef fingerprint(value):\n    return hashlib.sha256(json.dumps(value, sort_keys=True, default=str).encode()).hexdigest()\n\n\ndef file_hash(path):\n    h = hashlib.sha256()\n    with Path(path).open(\'rb\') as f:\n        for chunk in iter(lambda: f.read(8 << 20), b\'\'):\n            h.update(chunk)\n    return h.hexdigest()\n\n\ndef atomic_json(path, data):\n    path = Path(path); path.parent.mkdir(parents=True, exist_ok=True)\n    tmp = path.with_suffix(path.suffix + \'.tmp\')\n    tmp.write_text(json.dumps(data, indent=2, sort_keys=True, allow_nan=False, default=str))\n    tmp.replace(path)\n\n\ndef atomic_torch(path, data):\n    path = Path(path); path.parent.mkdir(parents=True, exist_ok=True)\n    tmp = path.with_suffix(\'.tmp\'); torch.save(data, tmp); tmp.replace(path)\n\n\ndef select_one(candidates, description):\n    candidates = sorted(set(Path(p).resolve() for p in candidates))\n    if len(candidates) != 1:\n        raise ValueError(f\'Set {description} explicitly: found {len(candidates)} candidates: {candidates}\')\n    return candidates[0]\n\n\ndef find_input_files(root, filename):\n    """Discover lightweight manifests without traversing hundreds of thousands of DICOMs."""\n    for current, dirs, files in os.walk(root):\n        dirs[:] = [d for d in dirs if d not in {\'train_series\',\'test_series\',\'cache\',\'features\',\'.git\',\'__pycache__\'}]\n        if filename in files:\n            yield Path(current)/filename\n\n\ndef discover_bundle(explicit=None, input_root=\'/kaggle/input\'):\n    if explicit:\n        root = Path(explicit)\n    else:\n        candidates = []\n        for p in find_input_files(input_root,\'dataset_index.json\'):\n            try:\n                index = json.loads(p.read_text())\n                if index.get(\'dataset_handle\') == \'gany24558/rsna-knee-normalized-training-data\':\n                    candidates.append(p.parent)\n            except (ValueError, OSError): pass\n        root = select_one(candidates, \'DATASET_ROOT\')\n    if not (root / \'dataset_index.json\').is_file():\n        raise FileNotFoundError(\'Attach gany24558/rsna-knee-normalized-training-data, with dataset_index.json\')\n    return root\n\n\ndef safe_member(archive, name):\n    part = PurePosixPath(name)\n    if part.is_absolute() or \'..\' in part.parts:\n        raise ValueError(f\'Unsafe archive member: {name}\')\n    member = archive.getmember(name)\n    if not member.isfile(): raise ValueError(f\'Not a regular member: {name}\')\n    return member\n\n\ndef load_bundle(root):\n    """Use only active, processed shards; never legacy archives or partial snapshots."""\n    root = Path(root); index = json.loads((root / \'dataset_index.json\').read_text())\n    identity = index[\'identity\']; expected = {f\'{i:03d}\' for i in range(int(identity[\'num_shards\']))}\n    if set(index[\'shards\']) != expected:\n        raise ValueError(f\'Preprocessing is incomplete. Missing shards: {sorted(expected-set(index["shards"]))}\')\n    frames, raw_reference, labels, prep_reference = [], None, None, None\n    for slot, entry in sorted(index[\'shards\'].items()):\n        if entry[\'status\'] != \'processed\': raise ValueError(f\'Shard {slot} is partial; resume preprocessing first\')\n        if Path(entry[\'file\']).name != entry[\'file\']: raise ValueError(\'Invalid package filename\')\n        package = root / entry[\'file\']\n        with tarfile.open(package, \'r:\') as arc:\n            info = json.load(arc.extractfile(safe_member(arc, \'publication_manifest.json\')))\n            if info[\'status\'] != \'processed\' or {k: info[k] for k in identity} != identity:\n                raise ValueError(\'Publication identity/status mismatch\')\n            prep = json.load(arc.extractfile(safe_member(arc, \'preprocessing_config.json\')))\n            if prep.get(\'run_id\') != identity[\'run_id\']: raise ValueError(\'Preprocessing config run mismatch\')\n            if prep_reference is not None and prep != prep_reference: raise ValueError(\'Preprocessing configs differ\')\n            prep_reference = prep\n            raw = arc.extractfile(safe_member(arc, \'study_labels_and_folds.csv\')).read()\n            if hashlib.sha256(raw).hexdigest() != identity[\'label_sha256\']:\n                raise ValueError(\'Label checksum mismatch\')\n            if raw_reference is not None and raw != raw_reference: raise ValueError(\'Labels differ across shards\')\n            raw_reference = raw\n            if labels is None:\n                labels = pd.read_csv(io.BytesIO(raw), dtype={ID: str, \'patient_group\': str})\n            frame = pd.read_csv(arc.extractfile(safe_member(arc, f\'training_series_shard_{int(slot):03d}.csv\')),\n                                dtype={ID: str, SID: str})\n            if len(frame) and not frame.status.eq(\'ok\').all(): raise ValueError(\'Training series contain failures\')\n            # Direct seek offsets avoid repeatedly scanning tar archives during extraction.\n            offsets, sizes = [], []\n            for member in frame.cache_path:\n                item = safe_member(arc, member); offsets.append(item.offset_data); sizes.append(item.size)\n            frame[\'archive_path\'] = str(package); frame[\'offset\'] = offsets; frame[\'nbytes\'] = sizes\n            frames.append(frame)\n    series = pd.concat(frames, ignore_index=True).sort_values([ID, SID]).reset_index(drop=True)\n    if series.empty or series.duplicated([ID, SID]).any(): raise ValueError(\'Empty or duplicated training series\')\n    if not labels[ID].is_unique: raise ValueError(\'Duplicate study labels\')\n    if not set(series[ID]).issubset(set(labels[ID])): raise ValueError(\'Series with no label record\')\n    validate_labels(labels)\n    eligible = labels.loc[labels[ID].isin(series[ID])].copy().sort_values(ID).reset_index(drop=True)\n    index = dict(index, preprocessing=prep_reference)\n    return series, eligible, labels, index\n\n\ndef validate_labels(labels):\n    required = {ID, \'fold\', \'patient_group\', \'has_gold\'}\n    required |= {t+s for t in TARGETS for s in [\'\', \'__mask\', \'__weight\', \'__gold\']}\n    if required - set(labels): raise ValueError(f\'Missing label columns: {sorted(required-set(labels))}\')\n    if labels.patient_group.isna().any() or labels.patient_group.str.strip().eq(\'\').any():\n        raise ValueError(\'Missing grouping identity\')\n    folds = pd.to_numeric(labels.fold, errors=\'raise\').to_numpy(float)\n    if not np.isfinite(folds).all() or not np.equal(folds, folds.astype(int)).all() or (folds < -1).any():\n        raise ValueError(\'Invalid fold values\')\n    if labels.groupby(\'patient_group\').fold.nunique().gt(1).any(): raise ValueError(\'A group crosses folds\')\n    for t in TARGETS:\n        for s in [\'__mask\', \'__gold\']:\n            if not labels[t+s].isin([0, 1]).all(): raise ValueError(\'Invalid label mask\')\n        m = labels[t+\'__mask\'].eq(1); g = labels[t+\'__gold\'].eq(1)\n        w = labels[t+\'__weight\'].to_numpy(float)\n        if not np.isfinite(w).all() or (w < 0).any(): raise ValueError(\'Invalid reliability weights\')\n        if not labels.loc[m, t].isin([0, 1]).all(): raise ValueError(\'Known target is not binary\')\n        if (g & ~m).any() or (g & labels[t+\'__weight\'].ne(1)).any(): raise ValueError(\'Invalid verified override\')\n    any_gold = labels[[t+\'__gold\' for t in TARGETS]].any(axis=1)\n    if not labels.has_gold.eq(any_gold.astype(int)).all(): raise ValueError(\'has_gold does not match per-target masks\')\n    if labels.loc[any_gold, \'fold\'].lt(0).any(): raise ValueError(\'Verified studies need validation folds\')\n\n\ndef audit_labels(labels):\n    rows = []\n    for fold, block in labels.groupby(\'fold\'):\n        for t in TARGETS:\n            m = block[t+\'__mask\'].eq(1) & block[t+\'__weight\'].gt(0)\n            g = block[t+\'__gold\'].eq(1)\n            rows.append(dict(fold=int(fold), target=t, studies=len(block), positive=int((m & block[t].eq(1)).sum()),\n                             negative=int((m & block[t].eq(0)).sum()), unknown=int((~m).sum()),\n                             verified_positive=int((g & block[t].eq(1)).sum()),\n                             verified_negative=int((g & block[t].eq(0)).sum())))\n    return pd.DataFrame(rows)\n\n\ndef read_native(record, expected_run):\n    with open(record[\'archive_path\'], \'rb\') as f:\n        f.seek(int(record[\'offset\'])); payload = f.read(int(record[\'nbytes\']))\n    if len(payload) != int(record[\'nbytes\']): raise ValueError(\'Truncated archive\')\n    with np.load(io.BytesIO(payload), allow_pickle=False) as z:\n        image = z[\'image\']; meta = json.loads(str(z[\'meta\'].item()))\n        support = np.unpackbits(z[\'valid_bits\'], count=image.size).reshape(image.shape).astype(bool)\n    if image.dtype != np.float16 or image.shape != (64, 320, 320) or list(image.shape) != meta[\'shape\']:\n        raise ValueError(\'Expected preprocessing contract float16 [64,320,320]\')\n    if meta.get(\'representation\') != \'native-plane-full-fov-v3\' or meta.get(\'run_id\') != expected_run:\n        raise ValueError(\'Wrong normalization identity/representation\')\n    if meta.get(ID) != record[ID] or meta.get(SID) != record[SID]: raise ValueError(\'Cache ID mismatch\')\n    if not np.isfinite(image).all() or image.min() < 0 or image.max() > 1: raise ValueError(\'Invalid pixel range\')\n    n = int(meta[\'selected_count\'])\n    if n <= 0 or n > len(image) or support[n:].any(): raise ValueError(\'Invalid acquired-slice mask\')\n    valid = np.flatnonzero(support[:n].reshape(n, -1).any(axis=1))\n    positions = np.asarray(meta[\'slice_positions_mm\'], dtype=np.float32)\n    spacing = np.asarray(meta[\'spacing_summary_drc_mm\'], dtype=np.float32)\n    if len(valid) != n or len(positions) != n or not np.isfinite(positions).all(): raise ValueError(\'Invalid slice positions\')\n    if n > 1 and not (np.diff(positions) > 0).all(): raise ValueError(\'Unordered slice positions\')\n    if spacing.shape != (3,) or not np.isfinite(spacing).all() or (spacing <= 0).any(): raise ValueError(\'Invalid spacing\')\n    return image, support, meta\n\n\ndef evaluate_metrics(y,p,gold):\n    rows=[]\n    for t,name in enumerate(TARGETS):\n        v=gold[:,t];truth=y[v,t];score=p[v,t];positive=int((truth==1).sum());negative=int((truth==0).sum())\n        auc=float(roc_auc_score(truth,score)) if positive and negative else None\n        ap=float(average_precision_score(truth,score)) if positive and negative else None\n        rows.append(dict(target=name,n=int(v.sum()),positive=positive,negative=negative,auroc=auc,average_precision=ap))\n    aucs=[r[\'auroc\'] for r in rows if r[\'auroc\'] is not None]\n    return dict(macro_auroc=float(np.mean(aucs)) if aucs else None,targets=rows,\n                note=\'Average precision uses sklearn; it is not trapezoidal PR-AUC.\')\n\n', 'cnn_runtime.py': '"""End-to-end ResNet34 study-level MRI training."""\nfrom pathlib import Path\nimport copy, json, time, hashlib, random\nimport numpy as np\nimport pandas as pd\nimport torch\nfrom torch import nn\nfrom torch.nn import functional as F\nfrom torch.utils.checkpoint import checkpoint\nfrom torchvision.models import resnet34, ResNet34_Weights\nimport knee_data as data\n\nclass ResNetKnee(nn.Module):\n    def __init__(self, pretrained=False, weights_path=None, dropout=.2):\n        super().__init__()\n        self.backbone=resnet34(weights=ResNet34_Weights.DEFAULT if pretrained and weights_path is None else None)\n        if weights_path:\n            self.backbone.load_state_dict(torch.load(weights_path,map_location=\'cpu\',weights_only=True),strict=True)\n        self.backbone.fc=nn.Identity()\n        self.head=nn.Sequential(nn.Dropout(dropout),nn.Linear(1024,12))\n        self.register_buffer(\'mean\',torch.tensor([.485,.456,.406]).view(1,3,1,1))\n        self.register_buffer(\'std\',torch.tensor([.229,.224,.225]).view(1,3,1,1))\n\n    def train(self,mode=True):\n        super().train(mode)\n        # Small study batches: retain pretrained BN running statistics; affine weights train.\n        for module in self.backbone.modules():\n            if isinstance(module,nn.BatchNorm2d):module.eval()\n        return self\n\n    def forward(self,series_images,microbatch=8):\n        if not series_images:raise ValueError(\'Empty study\')\n        device=self.mean.device; vectors=[]\n        for images in series_images:\n            features=[]\n            for start in range(0,len(images),microbatch):\n                x=images[start:start+microbatch].to(device)\n                x=(x-self.mean)/self.std\n                f=checkpoint(self.backbone,x,use_reentrant=False) if self.training else self.backbone(x)\n                features.append(f.float())\n            f=torch.cat(features)\n            vectors.append(torch.cat([f.mean(0),f.max(0).values]))\n        return self.head(torch.stack(vectors).mean(0,keepdim=True))\n\nclass EarlyStop:\n    def __init__(self,patience=3):\n        self.patience=patience;self.best=float(\'inf\');self.bad_epochs=0\n    def update(self,loss):\n        if not np.isfinite(loss):raise ValueError(\'Nonfinite validation loss\')\n        improved=loss<self.best\n        if improved:self.best=float(loss);self.bad_epochs=0\n        else:self.bad_epochs+=1\n        return improved,self.bad_epochs>=self.patience\n\ndef split_studies(labels,val_fraction=.2,seed=42):\n    from sklearn.model_selection import GroupShuffleSplit\n    data.validate_labels(labels)\n    splitter=GroupShuffleSplit(n_splits=1,test_size=val_fraction,random_state=seed)\n    train_ix,val_ix=next(splitter.split(labels,groups=labels.patient_group))\n    held=set(labels.iloc[val_ix].patient_group)\n    known=labels[[t+\'__mask\' for t in data.TARGETS]].to_numpy(bool)\n    weights=labels[[t+\'__weight\' for t in data.TARGETS]].to_numpy(float)\n    active=(known & (weights>0)).any(1)\n    train=labels.iloc[train_ix].loc[active[train_ix]].copy()\n    val=labels.iloc[val_ix].loc[labels.iloc[val_ix].has_gold.eq(1)].copy()\n    if train.empty or val.empty:raise ValueError(\'Need nonempty training and verified validation studies; inspect split\')\n    if set(train.patient_group)&held:raise ValueError(\'Group leakage\')\n    return train.reset_index(drop=True),val.reset_index(drop=True),held\n\ndef prepare_series(image,support,meta,cfg,rng=None):\n    n=int(meta[\'selected_count\']);count=cfg[\'slices_per_series\']\n    if n<=count:centers=np.arange(n)\n    elif rng is None:centers=np.linspace(0,n-1,count).round().astype(int)\n    else:centers=np.array([rng.choice(chunk) for chunk in np.array_split(np.arange(n),count)])\n    neighbors=np.clip(centers[:,None]+np.array([-1,0,1]),0,n-1)\n    x=torch.from_numpy(image[neighbors].astype(np.float32))\n    x=F.interpolate(x,size=(cfg[\'image_size\'],cfg[\'image_size\']),mode=\'bilinear\',align_corners=False,antialias=True)\n    # Full field of view, no flips or crops that might change anatomical interpretation.\n    if rng is not None:\n        x=(x*float(rng.uniform(.9,1.1))).clamp(0,1)\n    return x\n\ndef study_images(uid,by_study,run_id,cfg,rng=None):\n    records=by_study[uid]\n    if rng is not None and len(records)>cfg[\'max_train_series\']:\n        records=[records[i] for i in sorted(rng.choice(len(records),cfg[\'max_train_series\'],replace=False))]\n    images=[]\n    for record in records:\n        image,support,meta=data.read_native(record,run_id)\n        images.append(prepare_series(image,support,meta,cfg,rng))\n    return images\n\ndef row_targets(row,verified=False):\n    y=torch.tensor([[0 if pd.isna(row[t]) else float(row[t]) for t in data.TARGETS]],dtype=torch.float32)\n    mask=torch.tensor([[bool(row[t+(\'__gold\' if verified else \'__mask\')]) for t in data.TARGETS]])\n    w=torch.ones_like(y) if verified else torch.tensor([[float(row[t+\'__weight\']) for t in data.TARGETS]])\n    return y,mask,w\n\ndef masked_loss(logits,y,mask,weights):\n    w=mask.to(logits.dtype)*weights\n    if not (w>0).any():raise ValueError(\'Study has no supervised targets\')\n    return (F.binary_cross_entropy_with_logits(logits,y,reduction=\'none\')*w).sum()/w.sum()\n\nclass BudgetReached(Exception):pass\n\ndef check_budget(deadline):\n    if time.monotonic()>deadline:raise BudgetReached(\'Session time budget reached\')\n\n@torch.inference_mode()\ndef validate(model,val,by_study,run_id,cfg,device,deadline):\n    model.eval();zs=[];ys=[];masks=[]\n    for row in val.to_dict(\'records\'):\n        check_budget(deadline)\n        images=study_images(row[data.ID],by_study,run_id,cfg)\n        with torch.autocast(device.type,dtype=torch.float16,enabled=device.type==\'cuda\'):\n            z=model(images,cfg[\'microbatch\'])\n        zs.append(z.float().cpu());y,m,_=row_targets(row,True);ys.append(y);masks.append(m)\n    z,y,m=map(torch.cat,(zs,ys,masks))\n    if not torch.isfinite(z).all():raise ValueError(\'Nonfinite validation predictions\')\n    # Compute one loss over the full verified set, not a mean of unequal batch means.\n    loss=masked_loss(z,y,m,torch.ones_like(y)).item()\n    metrics=data.evaluate_metrics(y.numpy(),z.sigmoid().numpy(),m.numpy())\n    return loss,metrics,z.sigmoid().numpy()\n\ndef train_model(model,train,val,series,run_id,cfg,work,identity,resume=None):\n    work=Path(work);work.mkdir(parents=True,exist_ok=True)\n    device=next(model.parameters()).device\n    by={uid:g.sort_values(data.SID).to_dict(\'records\') for uid,g in series.groupby(data.ID)}\n    optimizer=torch.optim.Adam([{\'params\':model.backbone.parameters(),\'lr\':cfg[\'backbone_lr\']},\n                                {\'params\':model.head.parameters(),\'lr\':cfg[\'head_lr\']}])\n    scheduler=torch.optim.lr_scheduler.CosineAnnealingWarmRestarts(optimizer,T_0=10,T_mult=1,eta_min=1e-6)\n    scaler=torch.amp.GradScaler(\'cuda\',enabled=device.type==\'cuda\')\n    stop=EarlyStop(3);rng=np.random.default_rng(cfg[\'seed\']);history=[];start=0;best=None\n    if resume:\n        saved=torch.load(resume,map_location=device,weights_only=False) # only your own trusted checkpoint\n        if saved[\'identity\']!=identity:raise ValueError(\'Resume identity/config/data mismatch\')\n        model.load_state_dict(saved[\'state\']);optimizer.load_state_dict(saved[\'optimizer\'])\n        scheduler.load_state_dict(saved[\'scheduler\']);scaler.load_state_dict(saved[\'scaler\'])\n        rng.bit_generator.state=saved[\'rng\'];torch.set_rng_state(saved[\'torch_rng\'].cpu())\n        if device.type==\'cuda\':torch.cuda.set_rng_state_all([s.cpu() for s in saved[\'cuda_rng\']])\n        history=saved[\'history\'];start=saved[\'epoch\'];best=saved[\'best\']\n        stop.best=best[\'val_loss\'];stop.bad_epochs=saved[\'bad_epochs\']\n    deadline=time.monotonic()+cfg[\'max_hours\']*3600;reason=\'max_epochs\'\n    try:\n        if stop.bad_epochs>=3:reason=\'early_stopping\'\n        else:\n            for epoch in range(start,cfg[\'epochs\']):\n                model.train();records=train.to_dict(\'records\');order=rng.permutation(len(records));losses=[]\n                optimizer.zero_grad(set_to_none=True)\n                for step,idx in enumerate(order):\n                    check_budget(deadline)\n                    row=records[idx];images=study_images(row[data.ID],by,run_id,cfg,rng)\n                    y,m,w=[v.to(device) for v in row_targets(row)]\n                    window_start=(step//cfg[\'accumulate\'])*cfg[\'accumulate\']\n                    window_size=min(cfg[\'accumulate\'],len(order)-window_start)\n                    with torch.autocast(device.type,dtype=torch.float16,enabled=device.type==\'cuda\'):\n                        logits=model(images,cfg[\'microbatch\']);loss=masked_loss(logits,y,m,w)\n                    if not torch.isfinite(loss):raise ValueError(\'Nonfinite training loss\')\n                    scaler.scale(loss/window_size).backward();losses.append(float(loss.detach()))\n                    if (step+1)%cfg[\'accumulate\']==0 or step+1==len(order):\n                        scaler.unscale_(optimizer);torch.nn.utils.clip_grad_norm_(model.parameters(),1.)\n                        scaler.step(optimizer);scaler.update();optimizer.zero_grad(set_to_none=True)\n                    if (step+1)%100==0:print(f\'Epoch {epoch+1}: {step+1}/{len(order)} studies\',flush=True)\n                val_loss,metrics,pred=validate(model,val,by,run_id,cfg,device,deadline)\n                improved,done=stop.update(val_loss)\n                row=dict(epoch=epoch+1,train_loss=float(np.mean(losses)),val_loss=val_loss,\n                         macro_auroc=metrics[\'macro_auroc\'],bad_epochs=stop.bad_epochs)\n                history.append(row);print(row,flush=True)\n                if improved:\n                    best=dict(state_dict={k:v.detach().cpu().clone() for k,v in model.state_dict().items()},\n                              epoch=epoch+1,val_loss=val_loss,metrics=metrics)\n                    data.atomic_torch(work/\'best.pt\',best)\n                    frame=pd.DataFrame(pred,columns=data.TARGETS);frame.insert(0,data.ID,val[data.ID].tolist())\n                    frame.to_csv(work/\'validation_predictions_private.csv\',index=False)\n                scheduler.step(epoch+1)\n                pd.DataFrame(history).to_csv(work/\'history.csv\',index=False)\n                data.atomic_torch(work/\'last.pt\',dict(identity=identity,state=model.state_dict(),optimizer=optimizer.state_dict(),\n                    scheduler=scheduler.state_dict(),scaler=scaler.state_dict(),epoch=epoch+1,best=best,\n                    bad_epochs=stop.bad_epochs,history=history,rng=rng.bit_generator.state,\n                    torch_rng=torch.get_rng_state(),cuda_rng=torch.cuda.get_rng_state_all() if device.type==\'cuda\' else []))\n                if done:reason=\'early_stopping\';break\n    except BudgetReached:\n        reason=\'time_budget\';print(\'Time budget reached; resume from last.pt (last fully validated epoch).\',flush=True)\n    if best is None:raise RuntimeError(\'No completed validated epoch; no model will be exported or uploaded\')\n    model.load_state_dict(best[\'state_dict\']);model.eval()\n    data.atomic_json(work/\'training_status.json\',dict(reason=reason,best_epoch=best[\'epoch\'],best_val_loss=best[\'val_loss\']))\n    return best,history,reason\n\ndef export_model(model,best,cfg,index,identity,work,source_files):\n    package=Path(work)/\'model_package\';package.mkdir(exist_ok=True)\n    data.atomic_torch(package/\'model.pt\',dict(state_dict=best[\'state_dict\'],architecture=\'resnet34\',dropout=cfg[\'dropout\']))\n    for name,text in source_files.items():(package/name).write_text(text)\n    data.atomic_json(package/\'preprocessing_config.json\',index[\'preprocessing\'])\n    # Validate saved weights on synthetic image bags without training records.\n    loaded=ResNetKnee(pretrained=False,dropout=cfg[\'dropout\']).eval()\n    loaded.load_state_dict(torch.load(package/\'model.pt\',weights_only=True,map_location=\'cpu\')[\'state_dict\'],strict=True)\n    device=next(model.parameters()).device\n    gen=torch.Generator().manual_seed(71);images=[torch.rand(2,3,cfg[\'image_size\'],cfg[\'image_size\'],generator=gen)]\n    with torch.inference_mode():\n        expected=loaded(images).cpu();actual=model(images).float().cpu()\n    torch.testing.assert_close(actual,expected,rtol=1e-3,atol=1e-4)\n    data.atomic_torch(package/\'synthetic_smoke.pt\',dict(images=images,expected=expected))\n    import importlib.metadata\n    versions={k:importlib.metadata.version(k) for k in [\'torch\',\'torchvision\',\'numpy\',\'pandas\',\'scipy\',\'pydicom\',\'scikit-learn\']}\n    (package/\'requirements.txt\').write_text(\'\\n\'.join(f\'{k}=={v}\' for k,v in versions.items())+\'\\n\')\n    (package/\'README.md\').write_text(\'ResNet34 CNN knee MRI model. Load model.pt with cnn_runtime.ResNetKnee(pretrained=False).\\n\'\n        \'See manifest.json for native preprocessing, triplet slices, resize, normalization and aggregation.\\n\'\n        \'Not compatible with the DINOv2 submission notebook. Keep private. No labels or patient rows included.\\n\')\n    manifest=dict(schema_version=1,model_family=\'resnet34-study-mean-max-v1\',targets=data.TARGETS,configuration=cfg,\n        training_identity=identity,best_epoch=best[\'epoch\'],best_validation_loss=best[\'val_loss\'],\n        validation_metrics=best[\'metrics\'],environment=versions,\n        inference=dict(representation=\'adjacent-slice-triplet\',resize=\'bilinear antialias full-FOV\',\n        image_size=cfg[\'image_size\'],slices_per_series=cfg[\'slices_per_series\'],sampling=\'rounded linspace\',\n        series_selection=\'all usable, sorted SeriesInstanceUID\',slice_pool=\'mean+max\',series_pool=\'mean\',\n        outputs=\'12 logits; sigmoid once\',normalization_mean=[.485,.456,.406],normalization_std=[.229,.224,.225]),\n        files={str(p.relative_to(package)):data.file_hash(p) for p in package.rglob(\'*\') if p.is_file() and p.name!=\'manifest.json\'})\n    data.atomic_json(package/\'manifest.json\',manifest)\n    return package\n', 'native_preprocessing.py': 'from pathlib import Path\nimport hashlib, json, time, os, shutil, itertools\nimport numpy as np\nfrom scipy.ndimage import affine_transform, gaussian_filter\nimport pydicom\nfrom pydicom.pixels import apply_modality_lut\nID=\'StudyInstanceUID\'\nSID=\'SeriesInstanceUID\'\nAXES = {\n    \'Sagittal\': np.array([[1,0,0], [0,0,1], [0,-1,0]], float),\n    \'Coronal\': np.array([[0,0,1], [1,0,0], [0,-1,0]], float),\n    \'Axial\': np.array([[0,0,1], [0,1,0], [1,0,0]], float),\n}\ndef validate_and_extract_geometry(headers, expected_study, expected_series, plane, cfg):\n    """\n    Validates DICOM header geometry, checks for spatial consistency, checks \n    slice-spacing consistency within configured tolerances, and extracts the physical coordinate basis matrix.\n    \n    Parameters:\n        headers (list): List of pydicom dataset headers for a single series.\n        expected_study (str): Expected StudyInstanceUID for validation.\n        expected_series (str): Expected SeriesInstanceUID for validation.\n        plane (str): Expected anatomical plane (\'Sagittal\', \'Coronal\', \'Axial\').\n        cfg (dict): Pipeline configuration dictionary containing tolerances.\n        \n    Returns:\n        dict: Spatial metadata including slice order, origin, basis matrix, and warnings.\n    """\n    if plane not in AXES:\n        raise ValueError(f"Unknown plane: {plane}")\n    if len(headers) < 2:\n        raise ValueError("Need at least two spatial slices")\n\n    first = headers[0]\n    \n    # iop is an array of 6 floating-point numbers (standard medical imaging metadata):\n    # - The first 3 values (iop[:3]): Direction cosines for the image row.\n    # - The last 3 values (iop[3:]): Direction cosines for the image column.\n    iop = np.asarray(first.ImageOrientationPatient, float)\n    \n    spacing = np.asarray(first.PixelSpacing, float)\n\n    # 1. Validate initial orientation and spacing parameters\n    if iop.shape != (6,) or not np.isfinite(iop).all():\n        raise ValueError("Invalid orientation")\n    if spacing.shape != (2,) or not np.isfinite(spacing).all() or min(spacing) <= 0:\n        raise ValueError("Invalid pixel spacing")\n\n    col, row = iop[:3], iop[3:]\n    if (\n        abs(np.linalg.norm(col) - 1) > 1e-3\n        or abs(np.linalg.norm(row) - 1) > 1e-3\n        or abs(col @ row) > 1e-3\n    ):\n        raise ValueError("Orientation vectors are not orthonormal")\n\n    normal = np.cross(col, row)\n    photo = str(first.PhotometricInterpretation)\n    if photo not in {"MONOCHROME1", "MONOCHROME2"}:\n        raise ValueError("Only monochrome MRI supported")\n\n    shape = (int(first.Rows), int(first.Columns))\n    positions = []\n    sops = []\n\n    # 2. Iterate and validate consistency across all slices in the series\n    for h in headers:\n        if str(h.StudyInstanceUID) != expected_study or str(h.SeriesInstanceUID) != expected_series:\n            raise ValueError("DICOM study/series ID mismatch")\n        if int(getattr(h, "NumberOfFrames", 1)) != 1:\n            raise ValueError("Enhanced/multiframe DICOM unsupported")\n        if int(getattr(h, "SamplesPerPixel", 1)) != 1:\n            raise ValueError("Non-grayscale DICOM")\n        if (int(h.Rows), int(h.Columns)) != shape:\n            raise ValueError("Mixed dimensions within series")\n        if not np.allclose(h.ImageOrientationPatient, iop, atol=1e-4, rtol=0):\n            raise ValueError("Mixed orientations within series")\n        if not np.allclose(h.PixelSpacing, spacing, atol=1e-4, rtol=0):\n            raise ValueError("Mixed pixel spacing within series")\n        if str(h.PhotometricInterpretation) != photo:\n            raise ValueError("Mixed photometric interpretation")\n        \n        positions.append(np.asarray(h.ImagePositionPatient, float))\n        sops.append(str(h.SOPInstanceUID))\n\n    if len(set(sops)) != len(sops):\n        raise ValueError("Duplicate SOP instances")\n\n    positions = np.asarray(positions)\n    if positions.shape != (len(headers), 3) or not np.isfinite(positions).all():\n        raise ValueError("Missing/nonfinite slice position")\n\n    # 3. Sort slice positions and evaluate spatial gaps\n    order = np.argsort(positions @ normal, kind="stable")\n    positions = positions[order]\n    distances = positions @ normal\n    gaps = np.diff(distances)\n    dz = float(np.median(gaps))\n\n    if dz <= 0 or min(gaps) < 1e-3:\n        raise ValueError("Duplicate/non-spatial slice positions")\n    if np.max(abs(gaps - dz)) > max(0.05, cfg["gap_tolerance"] * dz):\n        raise ValueError("Irregular slice spacing / missing slices; not silently interpolated")\n\n    residual = positions - (positions[0] + np.arange(len(headers))[:, None] * dz * normal)\n    if np.linalg.norm(residual, axis=1).max() > 0.2:\n        raise ValueError("Nonparallel / shifted slice stack")\n\n    # 4. Compute anatomical obliquity and flag warnings if needed\n    angle = float(np.degrees(np.arccos(np.clip(abs(normal @ AXES[plane][:, 0]), 0, 1))))\n    nearest = min(AXES, key=lambda p: np.degrees(np.arccos(np.clip(abs(normal @ AXES[p][:, 0]), 0, 1))))\n    \n    review_warning = ""\n    if angle > cfg["max_obliquity_degrees"]:\n        review_warning = (\n            f"Orientation review: {angle:.1f} degrees from listed {plane}; "\n            f"warning threshold {cfg[\'max_obliquity_degrees\']:.1f}; nearest plane {nearest}. "\n            "Native acquisition retained; plane label unchanged."\n        )\n\n    # 5. Construct the final 3D coordinate basis matrix\n    basis = np.column_stack([normal * dz, row * spacing[0], col * spacing[1]])\n    \n    return dict(\n        order=order, origin=positions[0], basis=basis,\n        shape=(len(headers), *shape), photo=photo, angle=angle, dz=dz,\n        review_warning=review_warning, nearest_plane=nearest,\n        slice_thickness=float(getattr(first, "SliceThickness", 0) or 0)\n    )\n\n\ndef native_grid(g, plane, cfg):\n    """\n    Performs spatial normalization on individual scan slices. It aligns, scales, \n    and computes the 2D affine transformation matrices needed to map heterogeneous \n    scanner layouts onto a uniform, standardized output grid without arbitrary 3D rotations.\n    \n    Parameters:\n        g (dict): Spatial metadata dictionary extracted from geometry validation.\n        plane (str): Expected anatomical plane (\'Sagittal\', \'Coronal\', \'Axial\').\n        cfg (dict): Pipeline configuration dictionary containing target size and spacing.\n        \n    Returns:\n        tuple: (matrix, offset, basis, crop) used for downstream image resampling.\n    """\n    # 1. Select native in-plane axes without through-plane interpolation or arbitrary rotations\n    native = g[\'basis\'][:, 1:]\n    unit = native / np.linalg.norm(native, axis=0)\n    \n    # 2. Test valid axis permutations and sign flips to match the target anatomical orientation\n    choices = []\n    for perm in [(0, 1), (1, 0)]:\n        for signs in itertools.product([-1, 1], repeat=2):\n            directions = unit[:, perm] * np.array(signs)\n            score = float(np.sum(directions * AXES[plane][:, 1:]))\n            choices.append((score, directions))\n            \n    directions = max(choices, key=lambda x: x[0])[1]\n    shape = np.array(g[\'shape\'][1:])\n    \n    # 3. Calculate physical dimensions (extent) of the scan field of view (FOV)\n    corners = np.array(list(itertools.product(*[(0, n - 1) for n in shape])))\n    projections = (corners @ native.T) @ directions\n    extent = np.ptp(projections, axis=0)\n    \n    # 4. Determine pixel spacing, optionally scaling to fit the full FOV while preserving aspect ratio\n    spacing = float(cfg[\'spacing_mm\'])\n    if cfg.get(\'fit_full_fov\', False):\n        spacing = max(spacing, float(extent.max()) / (cfg[\'size\'] - 1))\n        \n    # 5. Compute the 2D affine transformation matrix:\n    #    The resampling matrix maps output pixel coordinates back to native source coordinates \n    #    using scaling, axis swaps and flips; offset supplies the translation. No arbitrary rotation is used.\n    basis = directions * spacing\n    matrix = np.linalg.pinv(native) @ basis\n    matrix[np.abs(matrix) < 1e-10] = 0\n    matrix = np.where(np.abs(matrix - np.rint(matrix)) < 1e-10, np.rint(matrix), matrix)\n    \n    # 6. Calculate centering offsets (translation part of the spatial alignment)\n    center = (shape - 1) / 2\n    offset = center - matrix @ np.full(2, (cfg[\'size\'] - 1) / 2)\n    offset[np.abs(offset) < 1e-10] = 0\n    offset = np.where(np.abs(offset - np.rint(offset)) < 1e-10, np.rint(offset), offset)\n    \n    # 7. Validate safety constraints to ensure excessive cropping does not occur\n    crop = np.maximum(0, 1 - (cfg[\'size\'] - 1) * spacing / np.maximum(extent, 1e-9))\n    if crop.max() > cfg[\'max_crop_fraction\'] + 1e-8:\n        raise ValueError(f\'Native FOV crop would be {crop.max():.1%}; increase size/spacing\')\n        \n    return matrix, offset, basis, crop\n\n\ndef decode_native_slice(path):\n    """\n    Loads an individual DICOM slice file, masks out background padding values, \n    applies the DICOM modality LUT or rescale parameters, and returns the processed float32 \n    image alongside a validity mask.\n    \n    Parameters:\n        path (str or Path): File path to the DICOM slice.\n        \n    Returns:\n        tuple: (image, valid_mask)\n            - image (np.ndarray): 32-bit floating-point array after modality scaling; MRI values need not have standardized physical units.\n            - valid_mask (np.ndarray): Boolean mask filtering out padding and non-finite values.\n    """\n    # 1. Load the DICOM file and validate that the pixel array is strictly 2D\n    ds = pydicom.dcmread(path)\n    raw = ds.pixel_array\n    if raw.ndim != 2:\n        raise ValueError("Expected single-frame 2D pixels")\n        \n    # 2. Mask explicit DICOM padding values; this does not identify all air or anatomical background\n    valid = np.ones(raw.shape, bool)\n    if hasattr(ds, "PixelPaddingValue"):\n        a = float(ds.PixelPaddingValue)\n        b = float(getattr(ds, "PixelPaddingRangeLimit", a))\n        valid &= (raw < min(a, b)) | (raw > max(a, b))\n        \n    # 3. Apply the modality LUT or rescale slope/intercept when supplied by the DICOM metadata\n    image = apply_modality_lut(raw, ds).astype(np.float32)\n    \n    # 4. Return the calibrated image and combine the padding mask with a finite-value check\n    return image, valid & np.isfinite(image)\n\n\ndef series_intensity_limits(paths, cfg):\n    """\n    Calculates robust global pixel intensity clipping bounds across an entire series \n    of DICOM slices using a memory-efficient, deterministic sampling strategy.\n    \n    Parameters:\n        paths (list): List of file paths to the DICOM slices in the series.\n        cfg (dict): Pipeline configuration containing sampling targets and percentile definitions.\n        \n    Returns:\n        tuple: (low, high) floating-point intensity limits for outlier clipping.\n    """\n    # 1. Determine a memory-efficient pixel sample budget per slice to prevent RAM spikes\n    per_slice = max(1, cfg[\'intensity_sample_pixels\'] // len(paths))\n    samples = []\n    \n    # 2. Iterate through each slice, decode, and extract evenly-spaced valid pixel samples\n    for path in paths:\n        image, valid = decode_native_slice(path)\n        values = image[valid]\n        if len(values):\n            # Select an evenly distributed subset of pixels using linear spacing\n            ix = np.linspace(0, len(values) - 1, min(per_slice, len(values))).astype(int)\n            samples.append(values[ix])\n            \n    # 3. Validate that a sufficient number of valid pixels were collected across the series\n    if not samples or sum(map(len, samples)) < 32:\n        raise ValueError(\'Insufficient valid pixels\')\n        \n    # 4. Compute lower and upper percentiles (configured here as 0.5th and 99.5th) to establish robust intensity bounds\n    low, high = np.percentile(np.concatenate(samples), cfg[\'percentiles\'])\n    \n    # 5. Verify that the calculated limits are finite numbers and form a non-degenerate range\n    if not np.isfinite([low, high]).all() or high <= low:\n        raise ValueError(\'Constant/invalid series intensity\')\n        \n    return float(low), float(high)\n\n\ndef normalize_native_slice(image, valid, photo, low, high, matrix, offset, cfg):\n    """\n    Normalizes pixel intensities, handles photometric inversion, applies \n    conditional anti-aliasing for downsampling, and resamples the 2D slice \n    onto the standardized output grid using an affine transformation.\n    \n    Parameters:\n        image (np.ndarray): 32-bit floating-point raw slice image.\n        valid (np.ndarray): Boolean validity mask for the image pixels.\n        photo (str): Photometric interpretation (e.g., \'MONOCHROME1\' or \'MONOCHROME2\').\n        low (float): Lower intensity clipping threshold.\n        high (float): Upper intensity clipping threshold.\n        matrix (np.ndarray): 2D affine transformation matrix.\n        offset (np.ndarray): Affine translation offset.\n        cfg (dict): Pipeline configuration containing target size, antialiasing flags, etc.\n        \n    Returns:\n        tuple: (output_slice, support_mask)\n            - output_slice (np.ndarray): 16-bit floating-point normalized and resampled image.\n            - support_mask (np.ndarray): Boolean support mask indicating valid resampled pixels.\n    """\n    # 1. Scale pixel values into a [0, 1] range using global intensity limits and clip outliers\n    image = np.clip((image - low) / (high - low), 0, 1).astype(np.float32)\n    \n    # 2. Invert pixel intensities if photometric interpretation is MONOCHROME1 (lower values are intended to display brighter)\n    if photo == \'MONOCHROME1\':\n        image = 1 - image\n        \n    # 3. Force invalid or background padding pixels to zero\n    image[~valid] = 0\n    \n    # 4. Apply conditional anti-aliasing via Gaussian smoothing for in-plane reductions (downsampling)\n    #    Anti-alias only in-plane reductions; no filtering is ever applied along depth.\n    sigma = .5 * np.sqrt(np.maximum(np.sum(matrix ** 2, axis=1) - 1, 0))\n    if cfg[\'antialias\'] and sigma.max() > 0:\n        numerator = gaussian_filter(image, sigma, mode=\'constant\', cval=0)\n        denominator = gaussian_filter(valid.astype(np.float32), sigma, mode=\'constant\', cval=0)\n        image = np.divide(numerator, denominator, out=np.zeros_like(numerator), where=denominator > 1e-6)\n        \n    # 5. Resample the image and validity mask onto the standardized target grid using affine transformations\n    shape = (cfg[\'size\'], cfg[\'size\'])\n    output = affine_transform(image, matrix, offset, output_shape=shape, order=1, mode=\'constant\', cval=0, prefilter=False)\n    support = affine_transform(valid.astype(np.float32), matrix, offset, output_shape=shape, order=1, mode=\'constant\', cval=0, prefilter=False) > .999\n    \n    # 6. Zero out unsupported regions, ensure all values are finite, clip, and cast to float16 for memory efficiency\n    output[~support] = 0\n    if not np.isfinite(output).all():\n        raise ValueError(\'Nonfinite output\')\n        \n    return np.clip(output, 0, 1).astype(np.float16), support\n\n\ndef select_native_indices(n, depth):\n    """\n    Selects a uniform subset of slice indices from a 3D scan volume to match \n    a target depth, ensuring both ends of the volume are always included.\n    \n    Parameters:\n        n (int): Total number of available native slices in the series.\n        depth (int): Target number of slices required by the pipeline configuration.\n        \n    Returns:\n        np.ndarray: 1D array of unique integer slice indices.\n    """\n    # 1. If the total available slices are fewer than or equal to the target depth, return all indices\n    if n <= depth:\n        return np.arange(n, dtype=int)\n        \n    # 2. Otherwise, use linear spacing to select evenly distributed actual slices across the volume\n    #    (guaranteeing inclusion of both endpoints without artificial duplication or interpolation)\n    indices = np.rint(np.linspace(0, n - 1, depth)).astype(int)\n    \n    # 3. Assert that all selected indices are unique to prevent overlapping or repeated slices\n    assert len(np.unique(indices)) == depth\n    \n    return indices\n\n\ndef process_series(folder, study, series, plane, cfg):\n    """\n    Orchestrates the end-to-end preprocessing pipeline for a 3D DICOM series folder, \n    handling file discovery, fast header parsing, spatial validation, orientation \n    normalization, and cache array generation.\n    \n    Parameters:\n        folder (str or Path): Directory path containing the DICOM series files.\n        study (str): Study identifier.\n        series (str): Series identifier.\n        plane (str): Target anatomical viewing plane.\n        cfg (dict): Pipeline configuration dictionary.\n        \n    Returns:\n        tuple: (normalized image array, valid-pixel mask, metadata dictionary).\n    """\n    # 1. Discover and sort all DICOM files in the directory; raise an error if none are found\n    paths = sorted(Path(folder).glob(\'*.dcm\'))\n    if not paths:\n        raise FileNotFoundError(f\'No DICOM slices in {folder}\')\n        \n    # 2. Efficiently parse DICOM headers across all files without loading heavy pixel data into RAM\n    headers = [pydicom.dcmread(p, stop_before_pixels=True) for p in paths]\n    \n    # 3. Extract and validate spatial geometry and orientation from the headers\n    g = validate_and_extract_geometry(headers, study, series, plane, cfg)\n    \n    # 4. Reverse slice order if needed so the native normal has nonnegative projection on the listed plane normal\n    if g[\'basis\'][:, 0] @ AXES[plane][:, 0] < 0:\n        g[\'origin\'] = g[\'origin\'] + g[\'basis\'][:, 0] * (len(paths) - 1)\n        g[\'basis\'][:, 0] *= -1\n        g[\'order\'] = g[\'order\'][::-1]\n        \n    # 5. Compute transformation matrices, offsets, and cropping parameters for the native grid\n    matrix, offset, basis, crop = native_grid(g, plane, cfg)\n    \n    # 6. Reorder file paths based on the validated spatial sequence and select target depth slice indices\n    ordered = [paths[i] for i in g[\'order\']]\n    indices = select_native_indices(len(paths), cfg[\'depth\'])\n    \n    # 7. Execute the final slice-by-slice processing and build the native cache arrays\n    return build_native_cache_arrays(ordered, headers, g, indices, matrix, offset, basis, crop, cfg)\n\n\ndef build_native_cache_arrays(paths, headers, g, indices, matrix, offset, basis, crop, cfg):\n    """\n    Executes the streaming core processing loop with an estimated-memory guard for selected slices, normalizes \n    and resamples each slice onto the target grid, validates valid-pixel coverage, \n    and compiles a comprehensive metadata audit log.\n    \n    Parameters:\n        paths (list): Sorted list of file paths to DICOM slices.\n        headers (list): List of parsed DICOM dataset headers.\n        g (dict): Validated spatial geometry dictionary.\n        indices (np.ndarray): Selected target slice indices.\n        matrix (np.ndarray): 2D affine transformation matrix.\n        offset (np.ndarray): Affine translation offset.\n        basis (np.ndarray): Spatial basis vectors.\n        crop (np.ndarray): Fraction of source extent cropped along each output in-plane axis.\n        cfg (dict): Pipeline configuration dictionary.\n        \n    Returns:\n        tuple: (output_tensor, support_mask, audit_metadata)\n    """\n    # 1. Enforce a defensive memory guardrail by estimating working memory consumption\n    estimate = int(np.prod(g[\'shape\'][1:])) * 64 + cfg[\'depth\'] * cfg[\'size\'] ** 2 * 6 + cfg[\'intensity_sample_pixels\'] * 16\n    if estimate > cfg[\'max_working_memory_gb\'] * 1e9:\n        raise MemoryError(f\'Estimated streaming working memory {estimate / 1e9:.2f} GB exceeds configured guard\')\n        \n    # 2. Compute global intensity clipping bounds and pre-allocate target 3D tensors\n    low, high = series_intensity_limits(paths, cfg)\n    shape = (cfg[\'depth\'], cfg[\'size\'], cfg[\'size\'])\n    output = np.zeros(shape, np.float16)\n    support = np.zeros(shape, bool)\n    \n    # 3. Initialize tracking collections and normal vectors for spatial provenance\n    origins = []\n    positions = []\n    sops = []\n    normal = g[\'basis\'][:, 0] / np.linalg.norm(g[\'basis\'][:, 0])\n    \n    # 4. Iteratively decode, normalize, resample, and validate each selected native slice\n    for slot, index in enumerate(indices):\n        image, valid = decode_native_slice(paths[index])\n        output[slot], support[slot] = normalize_native_slice(image, valid, g[\'photo\'], low, high, matrix, offset, cfg)\n        \n        # Verify that the slice meets the minimum required valid-pixel coverage fraction (not anatomical segmentation)\n        if support[slot].mean() < cfg[\'min_valid_fraction\']:\n            raise ValueError(\'Too little support in a selected native slice\')\n            \n        # Compute physical LPS origin and relative position along the normal axis\n        origin = g[\'origin\'] + g[\'basis\'][:, 0] * int(index) + g[\'basis\'][:, 1:] @ offset\n        origins.append(origin.tolist())\n        positions.append(float((origin - g[\'origin\']) @ normal))\n        sops.append(str(headers[g[\'order\'][index]].SOPInstanceUID))\n        \n    # 5. Compute slice spacing gaps and assemble the comprehensive audit metadata dictionary\n    gaps = np.diff(positions)\n    meta = dict(\n        shape=list(shape),\n        input_shape=list(g[\'shape\']),\n        representation=\'native-plane-full-fov-v3\',\n        intensity_low=low,\n        intensity_high=high,\n        intensity_limits_method=\'bounded deterministic all-slice sample\',\n        source_sop_order=[str(headers[i].SOPInstanceUID) for i in g[\'order\']],\n        selected_sop_uids=sops,\n        selected_source_indices=indices.tolist(),\n        selected_count=len(indices),\n        padded_count=cfg[\'depth\'] - len(indices),\n        retained_slice_fraction=len(indices) / len(paths),\n        slice_positions_mm=positions,\n        slice_gap_min_mm=float(gaps.min()),\n        slice_gap_max_mm=float(gaps.max()),\n        spacing_summary_drc_mm=[float(np.median(gaps)), *np.linalg.norm(basis, axis=0).tolist()],\n        requested_inplane_spacing_mm=float(cfg[\'spacing_mm\']),\n        fov_spacing_adjusted=bool(np.linalg.norm(basis, axis=0).max() > cfg[\'spacing_mm\'] + 1e-8),\n        output_slice_origins_lps=origins,\n        output_inplane_basis_lps=basis.tolist(),\n        input_origin_lps=g[\'origin\'].tolist(),\n        input_basis_lps=g[\'basis\'].tolist(),\n        source_slice_spacing_mm=g[\'dz\'],\n        source_slice_thickness_mm=g[\'slice_thickness\'],\n        obliquity_degrees=g[\'angle\'],\n        review_warning=g[\'review_warning\'],\n        nearest_plane=g[\'nearest_plane\'],\n        crop_fraction_rc=crop.tolist(),\n        estimated_working_memory_gb=estimate / 1e9,\n        valid_fraction=float(support[:len(indices)].mean()),\n        tensor_valid_fraction=float(support.mean()),\n        transfer_syntaxes=sorted({str(h.file_meta.TransferSyntaxUID) for h in headers})\n    )\n    \n    return output, support, meta\n'}
CODE_ROOT=Path('/kaggle/working/_resnet34_code')
CODE_ROOT.mkdir(parents=True,exist_ok=True)
for name,text in SOURCE_FILES.items():(CODE_ROOT/name).write_text(text)
sys.path.insert(0,str(CODE_ROOT))
import knee_data as data
import cnn_runtime as cnn
import numpy as np
import pandas as pd
import torch


## Validate data and split whole groups
Unknown labels never become negative labels. Verified labels must agree exactly with the competition train.csv. The split table and label counts are private diagnostics.

In [ ]:
if REQUIRE_GPU and not torch.cuda.is_available():raise RuntimeError('Enable a GPU')
if CFG['patience'] != 3:raise ValueError('This experiment requires patience=3')
if min(CFG[k] for k in ['microbatch','accumulate','slices_per_series','max_train_series','epochs'])<1:
    raise ValueError('Training sizes must be positive')
torch.manual_seed(CFG['seed']);np.random.seed(CFG['seed']);random.seed(CFG['seed'])
DEVICE=torch.device('cuda' if torch.cuda.is_available() else 'cpu')
ROOT=data.discover_bundle(DATASET_ROOT)
SERIES,LABELS,ALL_LABELS,INDEX=data.load_bundle(ROOT)
if INDEX['preprocessing'].get('implementation')!='ef1a2effbba938df32d512f010e3786904efc0cda3d04c95b074e29265018b5d':
    raise ValueError('Unexpected preprocessing implementation; audit before changing the contract')
if COMPETITION_ROOT is None:
    COMPETITION_ROOT=data.select_one([p.parent for p in data.find_input_files('/kaggle/input','train.csv')
        if (p.parent/'train_series.csv').is_file()],'COMPETITION_ROOT')
official=pd.read_csv(Path(COMPETITION_ROOT)/'train.csv',dtype={data.ID:str})
if not official[data.ID].is_unique or set(official[data.ID])!=set(ALL_LABELS[data.ID]):
    raise ValueError('Official training studies differ from generated-label dataset')
original=official.set_index(data.ID).loc[ALL_LABELS[data.ID]]
for target in data.TARGETS:
    gold=original[target].notna().to_numpy()
    if not np.array_equal(gold,ALL_LABELS[target+'__gold'].to_numpy(bool)):
        raise ValueError('Verified mask drift: '+target)
    if not np.array_equal(original[target].to_numpy()[gold],ALL_LABELS[target].to_numpy()[gold]):
        raise ValueError('Verified label drift: '+target)
TRAIN,VAL,HELD_GROUPS=cnn.split_studies(LABELS,CFG['val_fraction'],CFG['seed'])
IDENTITY=dict(configuration=CFG,dataset=INDEX['identity'],
              split_hash=data.fingerprint({'train':TRAIN[data.ID].tolist(),'held':sorted(HELD_GROUPS)}),
              source_hash=data.fingerprint(SOURCE_FILES),initialization='ImageNet ResNet34' if PRETRAINED_PATH is None else data.file_hash(PRETRAINED_PATH))
WORK=OUTPUT_ROOT/data.fingerprint(IDENTITY)[:16];WORK.mkdir(parents=True,exist_ok=True)
split=LABELS[[data.ID,'patient_group']].copy()
split['role']=np.where(split.patient_group.isin(HELD_GROUPS),'held_out','training_group')
split['used_for_validation']=split[data.ID].isin(VAL[data.ID])
split['used_for_training']=split[data.ID].isin(TRAIN[data.ID])
split.to_csv(WORK/'split_private.csv',index=False)
data.atomic_json(WORK/'run_identity.json',IDENTITY)
print('Training studies:',len(TRAIN),'Verified validation studies:',len(VAL))
print('All held-out studies:',split.role.eq('held_out').sum())
print('Grouping:', 'study only; patient separation unavailable' if LABELS.patient_group.eq(LABELS[data.ID]).all() else 'provided patient groups')
display(data.audit_labels(VAL))
MODEL=cnn.ResNetKnee(pretrained=RESUME_CHECKPOINT is None,weights_path=PRETRAINED_PATH,dropout=CFG['dropout']).to(DEVICE)


## Train the CNN and classifier
Adam uses 1e-5 for the pretrained CNN and 1e-4 for the head. Validation uses deterministic slices and all usable series. Each epoch scans all training studies. Checkpointing and accumulation bound GPU memory; BatchNorm running statistics stay fixed. If the session budget is reached, resume from last.pt in a new session. Only completed validated epochs are eligible for export.

In [ ]:
BEST,HISTORY,STOP_REASON=cnn.train_model(MODEL,TRAIN,VAL,SERIES,INDEX['identity']['run_id'],CFG,WORK,IDENTITY,RESUME_CHECKPOINT)
print('Stop reason:',STOP_REASON,'Best epoch:',BEST['epoch'],'Validation loss:',BEST['val_loss'])
display(pd.DataFrame(HISTORY))


## Export and check the best model
Only weights, runtime, transforms, configuration and aggregate metrics enter model_package. Split IDs, labels, validation predictions and optimizer checkpoints remain outside it.

In [ ]:
PACKAGE=cnn.export_model(MODEL,BEST,CFG,INDEX,IDENTITY,WORK,SOURCE_FILES)
print('Exported and reload-tested:',PACKAGE)


## Upload to the separate CNN Kaggle Model
Requires Internet and Kaggle authentication. New models default to private; existing models retain visibility. Re-running this cell creates a new version. If upload fails, the local package remains saved; retry only this cell, without retraining.

In [ ]:
if UPLOAD_MODEL:
    import kagglehub
    manifest=json.loads((PACKAGE/'manifest.json').read_text())
    for name,digest in manifest['files'].items():
        if data.file_hash(PACKAGE/name)!=digest:raise ValueError('Export changed: '+name)
    kagglehub.model_upload(MODEL_HANDLE,str(PACKAGE),version_notes=(
        f"ResNet34; grouped holdout; best epoch {BEST['epoch']}; verified BCE {BEST['val_loss']:.6f}; patience 3"))
    print('Upload accepted: https://www.kaggle.com/models/'+MODEL_HANDLE)
else:
    print('Upload disabled; package remains at',PACKAGE)
